# Análisis de ventas de una vidriería automotriz (datos simulados)

**Autor:** José León Tubay

En este proyecto analizo las ventas de un año de una vidriería de vidrios para autos (ficticia) usando **Python y pandas**.

> **Importante:** todos los datos de este cuaderno son **inventados (simulados)**. No pertenecen a ningún negocio real.

**Preguntas que voy a responder:**
1. ¿Qué productos generan más ingresos?
2. ¿En qué meses se vende más?
3. ¿Quiénes son los mejores clientes?
4. ¿Cuánto gasta un cliente, en promedio, por compra?

**Pasos:** crear los datos → revisarlos y limpiarlos → hacer cálculos → hacer gráficos → escribir conclusiones.

## Paso 1. Preparar las herramientas

- **pandas**: sirve para trabajar con tablas de datos (como un Excel, pero con código).
- **numpy**: sirve para generar números al azar.
- **matplotlib**: sirve para hacer gráficos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Para que los números se vean con 2 decimales
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Paso 2. Crear los datos inventados

Genero 600 ventas durante el año 2025. Cada venta tiene: fecha, cliente, producto, cantidad y precio.

- Uso una **semilla** (`42`) para que cada vez que corras el cuaderno salgan los mismos datos.
- Hago que unos meses vendan más que otros, sobre todo a fin de año, como pasa en muchos negocios.
- Además dejo a propósito **algunos errores** (precios vacíos y filas repetidas), porque los datos reales casi siempre vienen sucios y hay que limpiarlos.

In [ ]:
rng = np.random.default_rng(42)  # semilla: siempre salen los mismos datos

# producto: (precio mínimo, precio máximo, probabilidad de venderse)
productos = {
    "Parabrisas delantero":         (120, 250, 0.18),
    "Parabrisas posterior":         (90, 200, 0.10),
    "Ventana delantera izquierda":  (50, 110, 0.14),
    "Ventana delantera derecha":    (50, 110, 0.14),
    "Ventana posterior izquierda":  (45, 100, 0.10),
    "Ventana posterior derecha":    (45, 100, 0.10),
    "Ventolera":                    (30, 70, 0.12),
    "Vidrio lateral fijo":          (40, 90, 0.12),
}

n_ventas = 600

# Meses: algunos venden más que otros (los de fin de año tienen más peso)
peso_mes = np.array([0.9, 0.85, 1.1, 1.0, 0.95, 0.9, 1.0, 1.05, 1.0, 1.1, 1.15, 1.4])
peso_mes = peso_mes / peso_mes.sum()

meses = rng.choice(np.arange(1, 13), size=n_ventas, p=peso_mes)
dias = rng.integers(1, 29, size=n_ventas)
fechas = pd.to_datetime(pd.DataFrame({"year": 2025, "month": meses, "day": dias}))

clientes = [f"Cliente {i:02d}" for i in range(1, 31)]
prob_clientes = rng.dirichlet(np.ones(30) * 0.8)  # unos clientes compran más que otros

lista_productos = list(productos.keys())
prob_productos = [productos[p][2] for p in lista_productos]
producto_vendido = rng.choice(lista_productos, size=n_ventas, p=prob_productos)

precios = [round(rng.uniform(productos[p][0], productos[p][1]), 2) for p in producto_vendido]

df = pd.DataFrame({
    "fecha": fechas,
    "cliente": rng.choice(clientes, size=n_ventas, p=prob_clientes),
    "producto": producto_vendido,
    "cantidad": rng.integers(1, 5, size=n_ventas),
    "precio_unitario": precios,
})
df = df.sort_values("fecha").reset_index(drop=True)

# Ensuciamos un poco los datos a propósito (para practicar la limpieza)
df.loc[df.sample(6, random_state=1).index, "precio_unitario"] = np.nan   # 6 precios vacíos
df = pd.concat([df, df.sample(4, random_state=2)], ignore_index=True)    # 4 filas repetidas

df.head()

## Paso 3. Revisar los datos

Antes de calcular nada, hay que revisar que los datos estén bien. Reviso tres cosas:
1. Cuántas filas y columnas hay.
2. Si hay valores vacíos.
3. Si hay filas repetidas.

In [ ]:
print("Filas y columnas:", df.shape)
print()
print("Valores vacíos por columna:")
print(df.isna().sum())
print()
print("Filas repetidas:", df.duplicated().sum())

### Limpieza

- Elimino las filas repetidas (una venta no debe contarse dos veces).
- A los precios vacíos les pongo la **mediana** del precio de ese mismo producto (un valor razonable y que no se distorsiona con precios extremos).
- Creo dos columnas nuevas: `total` (cantidad × precio) y `mes`.

In [ ]:
df = df.drop_duplicates().copy()

mediana_por_producto = df.groupby("producto")["precio_unitario"].transform("median")
df["precio_unitario"] = df["precio_unitario"].fillna(mediana_por_producto)

df["total"] = df["cantidad"] * df["precio_unitario"]
df["mes"] = df["fecha"].dt.month

print("Valores vacíos después de limpiar:", int(df.isna().sum().sum()))
print("Filas repetidas después de limpiar:", int(df.duplicated().sum()))
print("Ventas finales:", len(df))
df.head()

## Paso 4. Hacerle preguntas a los datos

Aquí uso `groupby`, que sirve para **agrupar** filas (por producto, por mes, por cliente) y sumar o promediar.

In [ ]:
nombres_meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

# 1. Ingresos por producto
ventas_producto = df.groupby("producto")["total"].sum().sort_values(ascending=False)

# 2. Ingresos por mes
ventas_mes = df.groupby("mes")["total"].sum()
ventas_mes.index = nombres_meses

# 3. Mejores 5 clientes
top_clientes = df.groupby("cliente")["total"].sum().sort_values(ascending=False).head(5)

# 4. Gasto promedio por compra
ticket_promedio = df["total"].mean()

print("Ingresos por producto:")
print(ventas_producto)
print()
print("Mejores 5 clientes:")
print(top_clientes)
print()
print(f"Gasto promedio por compra: ${ticket_promedio:,.2f}")
print(f"Ingresos totales del año: ${df['total'].sum():,.2f}")

## Paso 5. Gráficos

In [ ]:
# Gráfico 1: ingresos por producto
ax = ventas_producto.sort_values().plot(kind="barh", figsize=(9, 5), color="#2a6f97")
ax.set_title("Ingresos por producto (2025)")
ax.set_xlabel("Ingresos ($)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: ingresos por mes
ax = ventas_mes.plot(kind="line", marker="o", figsize=(8, 4), color="#c1121f")
ax.set_title("Ingresos por mes (2025)")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos ($)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: mejores 5 clientes
ax = top_clientes.sort_values().plot(kind="barh", figsize=(8, 4), color="#588157")
ax.set_title("Mejores 5 clientes por compras (2025)")
ax.set_xlabel("Total comprado ($)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Paso 6. Conclusiones

Las conclusiones se calculan con código, así que se actualizan solas si cambian los datos.

In [ ]:
producto_top = ventas_producto.index[0]
pct_producto_top = ventas_producto.iloc[0] / ventas_producto.sum() * 100

mes_top = ventas_mes.idxmax()
mes_bajo = ventas_mes.idxmin()

pct_top5 = top_clientes.sum() / df["total"].sum() * 100

print(f"1. El producto que más ingresos genera es '{producto_top}' ({pct_producto_top:.1f}% del total).")
print(f"2. El mes con más ventas es {mes_top} y el más bajo es {mes_bajo}.")
print(f"3. Los 5 mejores clientes aportan el {pct_top5:.1f}% de los ingresos.")
print(f"4. Cada compra deja en promedio ${ticket_promedio:,.2f}.")

## Qué aprendí / herramientas usadas

- **pandas**: crear y limpiar tablas, `groupby`, columnas nuevas, valores vacíos y duplicados.
- **numpy**: generar datos aleatorios con semilla.
- **matplotlib**: gráficos de barras y de líneas.

**Ideas para mejorar este proyecto:**
- Usar datos reales (anonimizados) de un negocio.
- Predecir las ventas del siguiente mes.
- Hacer un panel interactivo con los resultados.